# Cyberbullying Detection — IEEE DataPort
## BERT + Attention Pooling (Final/Proposed Model)
Standalone notebook for your chosen final model — CLS-only pooling replaced with learned attention pooling over all token hidden states. No lexicon feature (dropped after ablation showed it hurt cyberstalking recall).

**Reproducibility fixes in this version:**
- Removed `langdetect`-based English filtering — it used a non-deterministic probabilistic detector that silently changed which rows survived cleaning on every run, shifting the train/val/test split underneath you. Dataset is Twitter data already curated for English-language cyberbullying research, so this step added risk without real benefit.
- Added `torch.backends.cudnn.deterministic = True` / `benchmark = False` to remove residual GPU-level run-to-run drift.

With both fixes, running this notebook twice on the same machine should now produce identical support counts and near-identical accuracy every time. Proper train/val/test separation (no leakage) and early stopping (patience=3) on validation loss, EPOCHS=10, are unchanged from before.

## 1. Setup

In [ ]:
!pip install -q emoji contractions imbalanced-learn codecarbon
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import time
import random

import emoji
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import RandomOverSampler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

seed_value = 2042
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

# Force fully deterministic GPU behavior (removes run-to-run drift from
# non-deterministic CUDA/cuDNN kernels, on top of the fixed seed above)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_style("whitegrid")
plt.rc("figure", autolayout=True)
plt.rc("axes", labelweight="bold", labelsize="large", titleweight="bold", titlepad=10)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Load IEEE DataPort dataset

In [ ]:
base_path = '/kaggle/input/datasets/sudhanshuchauhan29/cyber-bullying-dataset/'

df = pd.read_csv(base_path + 'IEEE Data Port.csv', encoding='latin1')
df = df.rename(columns={'Tweet': 'text', 'Class': 'sentiment'})
df = df[~df.duplicated()]
print(df.shape)
print(df['sentiment'].value_counts())

## 3. Tweet text deep cleaning (identical pipeline used throughout the project)

In [ ]:
def strip_emoji(text):
    if not isinstance(text, str):
        text = str(text)
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def strip_all_entities(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'\r|\n', ' ', text.lower())
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]', '', text)
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

def clean_hashtags(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    new_tweet = re.sub(r'(\s+#[\w-]+)+\s*$', '', tweet).strip()
    new_tweet = re.sub(r'#([\w-]+)', r'\1', new_tweet).strip()
    return new_tweet

def filter_chars(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join('' if ('$' in word) or ('&' in word) else word for word in text.split())

def remove_mult_spaces(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r"\s\s+", " ", text)

def expand_contractions(text):
    if not isinstance(text, str):
        text = str(text)
    return contractions.fix(text)

def remove_numbers(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'\d+', '', text)

def lemmatize(text):
    if not isinstance(text, str):
        text = str(text)
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w) for w in words)

def remove_short_words(text, min_len=2):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(w for w in text.split() if len(w) >= min_len)

def replace_elongated_words(text):
    if not isinstance(text, str):
        text = str(text)
    regex_pattern = r'\b(\w+)((\w)\3{2,})(\w*)\b'
    return re.sub(regex_pattern, r'\1\3\4', text)

def remove_repeated_punctuation(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[\?\.\!]+(?=[\?\.\!])', '', text)

def remove_extra_whitespace(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(text.split())

def remove_url_shorteners(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(
        r'(?:http[s]?://)?(?:www\.)?(?:bit\.ly|goo\.gl|t\.co|tinyurl\.com|tr\.im|is\.gd|'
        r'cli\.gs|u\.nu|url\.ie|tiny\.cc|alturl\.com|ow\.ly|bit\.do|adoro\.to)\S+', '', text)

def remove_spaces_tweets(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    return tweet.strip()

def remove_short_tweets(tweet, min_words=3):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    words = tweet.split()
    return tweet if len(words) >= min_words else ""

def clean_tweet(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    tweet = strip_emoji(tweet)
    tweet = expand_contractions(tweet)
    tweet = strip_all_entities(tweet)
    tweet = clean_hashtags(tweet)
    tweet = filter_chars(tweet)
    tweet = remove_mult_spaces(tweet)
    tweet = remove_numbers(tweet)
    tweet = lemmatize(tweet)
    tweet = remove_short_words(tweet)
    tweet = replace_elongated_words(tweet)
    tweet = remove_repeated_punctuation(tweet)
    tweet = remove_extra_whitespace(tweet)
    tweet = remove_url_shorteners(tweet)
    tweet = remove_spaces_tweets(tweet)
    tweet = ' '.join(tweet.split())
    return tweet

In [ ]:
df['text_clean'] = [clean_tweet(t) for t in df['text']]
print(f'{int(df["text_clean"].duplicated().sum())} duplicated cleaned tweets will be removed.')
df.drop_duplicates('text_clean', inplace=True)
df = df[df['text_clean'].str.len() > 0]
print(df['sentiment'].value_counts())

In [ ]:
sentiment = ["Cyberstalking", "Revenge Porn", "Doxing", "Sexual Harassment", "Slut Shaming"]

df['text_len'] = [len(t.split()) for t in df['text_clean']]
df = df[df['text_len'] < df['text_len'].quantile(0.995)]

df['sentiment'] = df['sentiment'].replace(
    {'Cyberstalking': 0, 'Revenge Porn': 1, 'Doxing': 2, 'Sexual Harassment': 3, 'Slut Shaming': 4}
)
print(f"Final dataset size: {len(df)}")

## 4. Train / Validation / Test split (no leakage — same as your fixed pipeline)

In [ ]:
X = df['text_clean'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed_value)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed_value)

print(f"Train: {len(X_train)} | Val: {len(X_valid)} | Test: {len(X_test)}")

## 5. Oversampling (training set only)

In [ ]:
ros = RandomOverSampler(random_state=seed_value)
X_train_res, y_train_res = ros.fit_resample(
    np.array(X_train).reshape(-1, 1), np.array(y_train).reshape(-1, 1))

X_train = X_train_res.flatten()
y_train = y_train_res.flatten()

(unique, counts) = np.unique(y_train, return_counts=True)
print("Class balance after oversampling:")
print(np.asarray((unique, counts)).T)

## 6. BERT Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
MAX_LEN = 128

def bert_tokenizer(data):
    input_ids, attention_masks = [], []
    for sent in data:
        encoded_sent = tokenizer(
            sent, add_special_tokens=True, max_length=MAX_LEN,
            padding='max_length', truncation=True, return_attention_mask=True
        )
        input_ids.append(encoded_sent['input_ids'])
        attention_masks.append(encoded_sent['attention_mask'])
    return torch.tensor(input_ids), torch.tensor(attention_masks)

train_inputs, train_masks = bert_tokenizer(X_train)
val_inputs, val_masks = bert_tokenizer(X_valid)
test_inputs, test_masks = bert_tokenizer(X_test)

## 7. DataLoaders

In [ ]:
train_labels = torch.tensor(y_train, dtype=torch.long)
val_labels = torch.tensor(y_valid, dtype=torch.long)
test_labels = torch.tensor(y_test, dtype=torch.long)

batch_size = 32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=batch_size)

val_data = TensorDataset(val_inputs, val_masks, val_labels)
val_dataloader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_dataloader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=batch_size)

print(len(next(iter(train_dataloader))), len(next(iter(val_dataloader))), len(next(iter(test_dataloader))))

## 8. Model — BERT + Attention Pooling (FINAL / PROPOSED MODEL)
CLS-only pooling is replaced with a learnable additive attention layer over **all** token hidden states (masked to ignore padding), producing a weighted-sum sentence representation instead of relying on just the [CLS] token.

In [ ]:
class Bert_Classifier_AttentionPool(nn.Module):
    def __init__(self, freeze_bert=False):
        super(Bert_Classifier_AttentionPool, self).__init__()
        n_hidden = 128
        n_output = 5

        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Attention pooling layer
        self.attention_weights = nn.Sequential(
            nn.Linear(768, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(n_hidden, n_output)
        )

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs[0]   # (batch, seq_len, 768)

        # Per-token attention score
        attn_scores = self.attention_weights(last_hidden_state).squeeze(-1)   # (batch, seq_len)
        # Mask out padding tokens before softmax
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=1).unsqueeze(-1)           # (batch, seq_len, 1)
        # Weighted sum over all tokens = attention-pooled sentence vector
        pooled_output = torch.sum(last_hidden_state * attn_probs, dim=1)       # (batch, 768)

        logits = self.classifier(pooled_output)
        return logits

In [ ]:
def initialize_model(epochs=10):
    bert_classifier = Bert_Classifier_AttentionPool(freeze_bert=False)
    bert_classifier.to(device)
    optimizer = AdamW(bert_classifier.parameters(), lr=5e-5, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    return bert_classifier, optimizer, scheduler

EPOCHS = 10
bert_classifier, optimizer, scheduler = initialize_model(epochs=EPOCHS)

## 9. Training loop (with early stopping on validation loss)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def bert_train(model, train_dataloader, val_dataloader, epochs=10, patience=3):
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    print("Start training...\n")
    for epoch_i in range(epochs):
        print("-"*10)
        print("Epoch : {}".format(epoch_i+1))
        print("-"*10)

        t0_epoch, t0_batch = time.time(), time.time()
        total_loss, batch_loss, batch_counts = 0, 0, 0

        model.train()
        for step, batch in enumerate(train_dataloader):
            batch_counts += 1
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)

            model.zero_grad()
            logits = model(b_input_ids, b_attn_mask)

            loss = loss_fn(logits, b_labels)
            batch_loss += loss.item()
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            if (step % 100 == 0 and step != 0) or (step == len(train_dataloader) - 1):
                time_elapsed = time.time() - t0_batch
                print(f"{step:^9} | {batch_loss / batch_counts:^12.6f} | {time_elapsed:^9.2f}")
                batch_loss, batch_counts = 0, 0
                t0_batch = time.time()

        avg_train_loss = total_loss / len(train_dataloader)

        ### EVALUATION on VALIDATION set ###
        model.eval()
        val_accuracy, val_loss = [], []

        for batch in val_dataloader:
            batch_input_ids, batch_attention_mask, batch_labels = tuple(t.to(device) for t in batch)
            with torch.no_grad():
                logits = model(batch_input_ids, batch_attention_mask)
            loss = loss_fn(logits, batch_labels)
            val_loss.append(loss.item())
            preds = torch.argmax(logits, dim=1).flatten()
            accuracy = (preds == batch_labels).cpu().numpy().mean() * 100
            val_accuracy.append(accuracy)

        val_loss = np.mean(val_loss)
        val_accuracy = np.mean(val_accuracy)
        time_elapsed = time.time() - t0_epoch

        print("-"*61)
        print(f"{'AVG TRAIN LOSS':^14} | {'VAL LOSS':^10} | {'VAL ACCURACY (%)':^17} | {'ELAPSED (s)':^9}")
        print(f"{avg_train_loss:^14.6f} | {val_loss:^10.6f} | {val_accuracy:^17.2f} | {time_elapsed:^9.2f}")
        print("-"*61 + "\n")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after epoch {epoch_i+1} (no val improvement for {patience} epochs).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model checkpoint (val loss = {best_val_loss:.6f}).")

    print("Training complete!")
    return model

bert_classifier = bert_train(bert_classifier, train_dataloader, val_dataloader, epochs=EPOCHS, patience=3)

## 10. Prediction + evaluation on TEST set

In [ ]:
import psutil
from codecarbon import EmissionsTracker

def bert_predict(model, test_dataloader, true_labels, device):
    tracker = EmissionsTracker()
    tracker.start()

    preds_list = []
    model.eval()

    process = psutil.Process()
    initial_memory = process.memory_info().rss / (1024 * 1024)
    start_time = time.time()

    for batch in test_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)

        with torch.no_grad():
            logit = model(batch_input_ids, batch_attention_mask)

        pred = torch.argmax(logit, dim=1).cpu().numpy()
        preds_list.extend(pred)

    end_time = time.time()
    inference_time = end_time - start_time
    final_memory = process.memory_info().rss / (1024 * 1024)
    memory_usage = final_memory - initial_memory
    cpu_usage = psutil.cpu_percent(interval=inference_time)
    emissions = tracker.stop()

    accuracy = accuracy_score(true_labels, preds_list)
    error_rate = 1 - accuracy

    print(f"Inference Time: {inference_time:.4f} seconds")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Error Rate: {error_rate:.4f}")
    print(f"Memory Usage: {memory_usage:.2f} MB")
    print(f"CPU Usage during Inference: {cpu_usage:.2f}%")
    print(f"Energy Consumption: {emissions} kWh")

    return preds_list

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

true_labels = []
for batch in test_dataloader:
    labels = batch[2].to(device).cpu().numpy()
    true_labels.extend(labels)

bert_preds = bert_predict(bert_classifier, test_dataloader, true_labels, device)

In [ ]:
print('Classification Report — BERT + Attention Pooling (Final Model):\n')
print(classification_report(y_test, bert_preds, target_names=sentiment, digits=3))

In [ ]:
def conf_matrix(y, y_pred, title, labels):
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    sns.heatmap(confusion_matrix(y, y_pred), annot=True, cmap="Purples", fmt='g',
                cbar=False, annot_kws={"size": 20}, xticklabels=labels, yticklabels=labels, ax=ax)
    plt.title(title, fontsize=20)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

conf_matrix(y_test, bert_preds, 'BERT + Attention Pooling\nConfusion Matrix (IEEE DataPort)', sentiment)

## Next step
Once you're happy with this, we adapt this exact notebook for the **Kaggle dataset (6 classes)**: swap the data-loading cell, extend `sentiment` to 6 labels, set `n_output=6` in `Bert_Classifier_AttentionPool`, and re-run. Architecture and training loop stay identical.